
# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 ตัวอย่างที่ 1: ใช้ NumPy คำนวณข้อมูล NDVI จากภาพ Raster


In [ ]:

import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.7, 0.8, 0.6], [0.9, 0.5, 0.3], [0.4, 0.7, 0.2]])
red = np.array([[0.3, 0.4, 0.2], [0.5, 0.3, 0.1], [0.2, 0.3, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)


ค่า NDVI:
[[0.4        0.33333333 0.5       ]
 [0.28571429 0.25       0.5       ]
 [0.33333333 0.4        0.33333333]]



## 🔹 ตัวอย่างที่ 2: ใช้ NumPy คำนวณค่าเฉลี่ยและค่ามากสุดของ NDVI


In [ ]:

print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")


ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25



## 🔹 ตัวอย่างที่ 3: ใช้ GeoPandas โหลดและวิเคราะห์ข้อมูล Shapefile


In [33]:

import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
gdf = gpd.read_file("/content/drive/MyDrive/234/tha_admbnda_adm1_rtsd_20190221")

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())

   Shape_Leng  Shape_Area        ADM1_EN        ADM1_TH ADM1_PCODE ADM1_REF  \
0    3.927244    0.275313  Amnat Charoen     อำนาจเจริญ       TH37     None   
1    1.739908    0.079210      Ang Thong        อ่างทอง       TH15     None   
2    2.417227    0.131339        Bangkok  กรุงเทพมหานคร       TH10     None   
3    4.414998    0.340784      Bueng Kan         บึงกาฬ       TH38     None   
4    8.701860    0.844537       Buri Ram      บุรีรัมย์       TH31     None   

  ADM1ALT1EN ADM1ALT2EN ADM1ALT1TH ADM1ALT2TH   ADM0_EN    ADM0_TH ADM0_PCODE  \
0       None       None       None       None  Thailand  ประเทศไทย         TH   
1       None       None       None       None  Thailand  ประเทศไทย         TH   
2       None       None       None       None  Thailand  ประเทศไทย         TH   
3       None       None       None       None  Thailand  ประเทศไทย         TH   
4       None       None       None       None  Thailand  ประเทศไทย         TH   

        date    validOn validTo  \
0 2


## 🔹 ตัวอย่างที่ 4: คำนวณพื้นที่ของแต่ละจังหวัด


In [48]:
# ตรวจสอบค่า CRS (Coordinate Reference System)
print(f"Original CRS: {gdf.crs}")

# ตรวจสอบว่าเป็น Geographic CRS หรือไม่
if gdf.crs and gdf.crs.is_geographic:
    print("Reprojecting to an equal-area projected CRS (EPSG:3857) for accurate area calculation...")
    gdf_proj = gdf.to_crs(epsg=3857)
else:
    gdf_proj = gdf.copy()

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร จากนั้นแปลงเป็นตารางกิโลเมตร)
gdf["area_sqkm"] = gdf_proj.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 5 อันดับแรกที่ใหญ่ที่สุด
print(gdf.nlargest(5, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

Original CRS: EPSG:4326
Reprojecting to an equal-area projected CRS (EPSG:3857) for accurate area calculation...
        ADM1_TH     area_sqkm
9     เชียงใหม่  24880.095257
28   นครราชสีมา  22317.931970
15    กาญจนบุรี  20891.294974
68          ตาก  18946.947972
71  อุบลราชธานี  16720.834730



## 🔹 ตัวอย่างที่ 5: ใช้ GeoPandas ทำ Spatial Join


In [93]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("/content/drive/MyDrive/234/thailand_province_amphoe")

# แสดงชื่อคอลัมน์ของ gdf_districts
print("Columns in gdf_districts:")
print(gdf_districts.columns)
print("\nFirst 5 rows of gdf_districts:")
display(gdf_districts.head())

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
# print(gdf_joined[['ADM2_TH_left', 'ADM1_TH_right']].head())


Columns in gdf_districts:
Index(['Shape_Leng', 'Shape_Area', 'ADM2_EN', 'ADM2_TH', 'ADM2_PCODE',
       'ADM1_EN', 'ADM1_TH', 'ADM1_PCODE', 'ADM0_EN', 'ADM0_TH', 'ADM0_PCODE',
       'geometry'],
      dtype='object')

First 5 rows of gdf_districts:


,Shape_Leng,Shape_Area,ADM2_EN,ADM2_TH,ADM2_PCODE,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM0_EN,ADM0_TH,ADM0_PCODE,geometry
0,0.085417,0.000450,Phra Nakhon,พระนคร,TH1001,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.50075 13.74107, 100.49981 13.738..."
1,0.134132,0.000950,Dusit,ดุสิต,TH1002,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.52 13.8, 100.52074 13.79992, 100..."
2,0.676342,0.019859,Nong Chok,หนองจอก,TH1003,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.91398 13.94621, 100.91395 13.945..."
3,0.085886,0.000337,Bang Rak,บางรัก,TH1004,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.51703 13.71808, 100.51703 13.718..."
4,0.301722,0.003415,Bang Khen,บางเขน,TH1005,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.60999 13.89078, 100.60997 13.890..."



# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: ใช้ NumPy คำนวณค่า **Mean, Max, Min** ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง
2. **แบบฝึกหัด 2**: ใช้ GeoPandas โหลด **Shapefile** ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด
3. **แบบฝึกหัด 3**: ใช้ GeoPandas ทำ **Spatial Join** ระหว่างข้อมูลจังหวัดและอำเภอ
4. **แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


1.แบบฝึกหัด 1: ใช้ NumPy คำนวณค่า Mean, Max, Min ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง

In [42]:
import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.8, 0.9, 0.7], [0.6, 0.7, 0.4], [0.5, 0.8, 0.3]])
red = np.array([[0.2, 0.3, 0.1], [0.4, 0.2, 0.2], [0.1, 0.2, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)

print(f"\nค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")

ค่า NDVI:
[[0.6        0.5        0.75      ]
 [0.2        0.55555556 0.33333333]
 [0.66666667 0.6        0.5       ]]

ค่าเฉลี่ย NDVI: 0.52
ค่า NDVI สูงสุด: 0.75
ค่า NDVI ต่ำสุด: 0.20


2.แบบฝึกหัด 2: ใช้ GeoPandas โหลด Shapefile ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
import geopandas as gpd
import pandas as pd # Import pandas to set display options

# Set pandas display options to show all rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
gdf = gpd.read_file("/content/drive/MyDrive/234/tha_admbnda_adm1_rtsd_20190221")
# แสดงข้อมูลทั้งหมด 77 จังหวัด
print("ข้อมูล GeoDataFrame ของจังหวัดทั้งหมด:")
print(gdf)

# ตรวจสอบค่า CRS (Coordinate Reference System)
print(f"\nOriginal CRS: {gdf.crs}")

# ตรวจสอบว่าเป็น Geographic CRS หรือไม่
if gdf.crs and gdf.crs.is_geographic:
    print("Reprojecting to an equal-area projected CRS (EPSG:3857) for accurate area calculation...")
    gdf_proj = gdf.to_crs(epsg=3857) # Reproject to Web Mercator (meters)
else:
    gdf_proj = gdf.copy()

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร จากนั้นแปลงเป็นตารางกิโลเมตร)
gdf["area_sqkm"] = gdf_proj.geometry.area / 1e6

# แสดงพื้นที่จังหวัดทั้งหมด 77 จังหวัด
print("\nพื้นที่ของแต่ละจังหวัด (ตารางกิโลเมตร):")
print(gdf.nlargest(77, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

# Reset pandas display options to default after printing (optional but good practice)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

ข้อมูล GeoDataFrame ของจังหวัดทั้งหมด:
    Shape_Leng  Shape_Area                   ADM1_EN          ADM1_TH  \
0     3.927244    0.275313             Amnat Charoen       อำนาจเจริญ   
1     1.739908    0.079210                 Ang Thong          อ่างทอง   
2     2.417227    0.131339                   Bangkok    กรุงเทพมหานคร   
3     4.414998    0.340784                 Bueng Kan           บึงกาฬ   
4     8.701860    0.844537                  Buri Ram        บุรีรัมย์   
5     4.941453    0.431484              Chachoengsao       ฉะเชิงเทรา   
6     2.896316    0.209078                  Chai Nat           ชัยนาท   
7     7.055513    1.065319                Chaiyaphum          ชัยภูมิ   
8     5.130273    0.533316               Chanthaburi         จันทบุรี   
9    13.346345    1.900546                Chiang Mai        เชียงใหม่   
10    6.991487    0.999021                Chiang Rai         เชียงราย   
11    6.091786    0.375630                 Chon Buri           ชลบุรี   
12    6.7510

3.แบบฝึกหัด 3: ใช้ GeoPandas ทำ Spatial Join ระหว่างข้อมูลจังหวัดและอำเภอ

In [92]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("/content/drive/MyDrive/234/thailand_province_amphoe")

# แสดงชื่อคอลัมน์ของ gdf_districts
print("Columns in gdf_districts:")
print(gdf_districts.columns)
print("\nFirst 5 rows of gdf_districts:")
display(gdf_districts.head())

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
# print(gdf_joined[['ADM2_TH_left', 'ADM1_TH_right']].head())

Columns in gdf_districts:
Index(['Shape_Leng', 'Shape_Area', 'ADM2_EN', 'ADM2_TH', 'ADM2_PCODE',
       'ADM1_EN', 'ADM1_TH', 'ADM1_PCODE', 'ADM0_EN', 'ADM0_TH', 'ADM0_PCODE',
       'geometry'],
      dtype='object')

First 5 rows of gdf_districts:


,Shape_Leng,Shape_Area,ADM2_EN,ADM2_TH,ADM2_PCODE,ADM1_EN,ADM1_TH,ADM1_PCODE,ADM0_EN,ADM0_TH,ADM0_PCODE,geometry
0,0.085417,0.000450,Phra Nakhon,พระนคร,TH1001,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.50075 13.74107, 100.49981 13.738..."
1,0.134132,0.000950,Dusit,ดุสิต,TH1002,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.52 13.8, 100.52074 13.79992, 100..."
2,0.676342,0.019859,Nong Chok,หนองจอก,TH1003,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.91398 13.94621, 100.91395 13.945..."
3,0.085886,0.000337,Bang Rak,บางรัก,TH1004,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.51703 13.71808, 100.51703 13.718..."
4,0.301722,0.003415,Bang Khen,บางเขน,TH1005,Bangkok,กรุงเทพมหานคร,TH10,Thailand,ประเทศไทย,TH,"POLYGON ((100.60999 13.89078, 100.60997 13.890..."


4.แบบฝึกหัด 4: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด

In [56]:
# ตรวจสอบว่า gdf (GeoDataFrame ของจังหวัด)
if 'gdf' not in locals():
    print("Error: 'gdf' (GeoDataFrame ของจังหวัด) ไม่พบ. กรุณารันเซลล์โหลดข้อมูลจังหวัดก่อนหน้านี้.")
else:
    # สร้างค่า NDVI เฉลี่ยจำลองสำหรับแต่ละจังหวัด
    # สมมติว่ามีค่า NDVI เฉลี่ยระหว่าง 0.1 ถึง 0.9
    np.random.seed(42)
    gdf['avg_ndvi'] = np.random.uniform(0.1, 0.9, size=len(gdf))

    # หาจังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด
    province_max_ndvi = gdf.loc[gdf['avg_ndvi'].idxmax()]

    print("จังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด:")
    print(f"  ชื่อจังหวัด (ไทย): {province_max_ndvi['ADM1_TH']}")
    print(f"  ค่า NDVI เฉลี่ย: {province_max_ndvi['avg_ndvi']:.2f}")


จังหวัดที่มีค่า NDVI เฉลี่ยสูงสุด:
  ชื่อจังหวัด (ไทย): ตรัง
  ค่า NDVI เฉลี่ย: 0.89
